In [1]:
from client import *
from common_imports import *
from collections.abc import Awaitable, Callable
from agent_framework import (AgentMiddleware,
                             AgentContext, 
                             ChatContext, 
                             FunctionInvocationContext, 
                             FunctionMiddleware,)
import time

In [2]:
async def logging_function_middleware(
    context: FunctionInvocationContext,
    call_next: Callable[[], Awaitable[None]],
) -> None:
    """Function middleware that logs function calls and results."""
    logger.info(f"[🪵 Logging][ Function Middleware] Calling {context.function.name} with args: {context.arguments}")

    await call_next()

    logger.info(f"[🪵 Logging][ Function Middleware] {context.function.name} returned: {context.result}")


async def logging_chat_middleware(
    context: ChatContext,
    call_next: Callable[[], Awaitable[None]],
) -> None:
    """Chat middleware that logs AI interactions."""
    logger.info(f"[💬 Logging][ Chat Middleware] Sending {len(context.messages)} messages to AI")

    await call_next()

    logger.info("[💬 Logging][ Chat Middleware] AI response received")

In [3]:
async def timing_agent_middleware(
    context: AgentContext,
    call_next: Callable[[], Awaitable[None]],
) -> None:
    """Agent middleware that logs execution timing."""
    start = time.perf_counter()
    logger.info("[⏲️ Timing][ Agent Middleware] Starting agent execution")

    await call_next()

    elapsed = time.perf_counter() - start
    logger.info(f"[⏲️ Timing][ Agent Middleware] Execution completed in {elapsed:.2f}s")

In [4]:
def get_weather(
    city: Annotated[str, Field(description="The city to get the weather for.")],
) -> dict:
    """Returns weather data for a given city, a dictionary with temperature and description."""
    logger.info(f"Getting weather for {city}")
    if random.random() < 0.05:
        return {
            "temperature": 72,
            "description": "Sunny",
        }
    else:
        return {
            "temperature": 60,
            "description": "Rainy",
        }


def get_activities(
    city: Annotated[str, Field(description="The city to get activities for.")],
    date: Annotated[str, Field(description="The date to get activities for in format YYYY-MM-DD.")],
) -> list[dict]:
    """Returns a list of activities for a given city and date."""
    logger.info(f"Getting activities for {city} on {date}")
    return [
        {"name": "Hiking", "location": city},
        {"name": "Beach", "location": city},
        {"name": "Museum", "location": city},
    ]


def get_current_date() -> str:
    """Gets the current date from the system and returns as a string in format YYYY-MM-DD."""
    logger.info("Getting current date")
    return datetime.now().strftime("%Y-%m-%d")


In [5]:
agent = Agent(
    client=client,
    instructions=(
        "You help users plan their weekends and choose the best activities for the given weather. "
        "If an activity would be unpleasant in weather, don't suggest it. Include date of the weekend in response."
    ),
    tools=[get_weather, get_activities, get_current_date],
    middleware=[
        # Agent-level middleware applied to ALL runs
        logging_chat_middleware,
        logging_function_middleware,
        timing_agent_middleware,
    ],
)

In [6]:
response = await agent.run("hii what can I do this weekend in San Francisco?")
print(response)

[⏲️ Timing][ Agent Middleware] Starting agent execution
[💬 Logging][ Chat Middleware] Sending 1 messages to AI
[💬 Logging][ Chat Middleware] AI response received
[🪵 Logging][ Function Middleware] Calling get_current_date with args: {}
Getting current date
[🪵 Logging][ Function Middleware] get_current_date returned: [<agent_framework._types.Content object at 0x000001B733AA4980>]
[💬 Logging][ Chat Middleware] Sending 1 messages to AI
[💬 Logging][ Chat Middleware] AI response received
[🪵 Logging][ Function Middleware] Calling get_weather with args: {'city': 'San Francisco'}
Getting weather for San Francisco
[🪵 Logging][ Function Middleware] get_weather returned: [<agent_framework._types.Content object at 0x000001B733AA7A10>]
[🪵 Logging][ Function Middleware] Calling get_activities with args: {'city': 'San Francisco', 'date': '2026-07-11'}
Getting activities for San Francisco on 2026-07-11
[🪵 Logging][ Function Middleware] get_activities returned: [<agent_framework._types.Content object at

This weekend in San Francisco (July 11-12, 2026), the weather is expected to be rainy with a temperature of 60°F. Considering the wet conditions, outdoor activities might not be ideal, but indoor options work well. Here are my recommendations:

### Suggested Activities:
1. **Visit a Museum**  
   Explore one of San Francisco's renowned museums like the San Francisco Museum of Modern Art (SFMOMA) or the Exploratorium. These are perfect for a rainy day.  

### Activities to Avoid:
- **Hiking**: Trails may be muddy and slippery due to rain, making it less enjoyable and safe.
- **Beach**: Rain and cool temperatures make beaches less ideal for relaxation this weekend.

Enjoy your weekend!
